# Datamine MODCONF Process Example

This notebook demonstrates how to configure and run the **`modconf`** process wrapper in `dmstudio`.

## Process Description

## MODCONF

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

The MODCONF process is part of the Conditional Simulation module, and calculates the confidence of the tonnage and grade resource estimates of a block model.

The main input is a simulated model created by the [CSMODEL](<csmodel.md>) process. The output includes two tables: the main one showing the EType estimate, and a range of percentiles for grade, tonnage and metal content for a set of user-defined cutoff grades. A set of graphs based on the output tables is also created.

The input model can simply be a subset of cells from a simulated model \- the modeled volume representing a pushback, or the material mined in a week or month, for example. The following observations could potentially be derived from the results of this process:

* Based on a 10x10x10m SMU, and a 1.0 g/t cutoff, the best estimate of the modeled resource is 1.38 mt at 2.68 g/t.
* There is a 90% probability that the grade of the resource will exceed 2.61 g/t, and a 90% probability that it will be less than 2.74 g/t.
* There is an 80% probability that the contained metal will exceed 3.66 million grams.

### Input Model File

Typically, the input model file is the simulated model (SIMMOD), created using the CSMODEL process. The following fields are required:

* _SIMi_ i=1,N where N is the maximum number of realizations. Each SIMi value represents the simulated grade of a model cell.
* _ETYPE_ The average of all realizations in a cell.

**Note** : CSMODEL creates additional fields, not required by MODCONF.

### Cutoff Grades

Results are presented for a set of regular cutoff grades that are selected using the CUTINT and CUTMAX parameters:

* _CUTINT_ Defines the interval between successive cutoffs, starting at zero.
* _CUTMAX_ he maximum cutoff to be considered.

* **Note** (If the maximum cell value of any realization is less than CUTMAX, then CUTMAX is automatically reduced so that all realizations have at least one value above the maximum.):

### Percentiles

Results are presented for a set of regular percentiles that are selected using the PCINT parameter. This parameter defines the interval between successive percentiles, and is restricted to '2.5', '5', '10', '20' or '25'.

### Other Parameters

* _DENSITY_ Used for tonnage calculations.
* _FACTOR_ For graphs with tonnage or metal content values annotated on their axes, these values are divided by the FACTOR parameter prior to plotting, in order to reduce the amount of annotation.

### EType Estimates

In addition to results for simulations, the tables and graphs include results for two EType estimates, as follows:

* _ETYPE1_ The average of all simulations within a cell. This is the ETYPE field that must be available in the input SIMMOD model file, and is processed and displayed in a similar way to the simulated grades for each cell.
* _ETYPE2_ The average of all simulations for the entire model. For a cutoff of 0, values for ETYPE1 and ETYPE2 will be identical; however, for cutoffs greater than 0, the ETYPE2 grade will be greater than the ETYPE1 grade.

### Output Confidence Tables

The file **CONF_TBL** requires a filename template to be specified. Two file names are then created automatically from the template by adding the characters '_1' and '_2'. For example, if the template name CONF_MODA is specified, then two files are created, as follows:

* _CONF_MODA_1_ Grade, volume, tonnes and metal for every combination of cutoff and simulation.
* _CONF_MODA_2_ Grade, metal and tonnes by cutoff for each percentile.

A portion of a table is shown below that has been calculated using the following parameters:

* Ten simulations are included in the input SIMMOD model.

* **Note** (This number of simulations is too few for a conclusive study, and is for example purposes only.):

* The model cell size is 10x10x10.
* **CUTINT** =1, **CUTMAX** =8: the results for nine cutoff grades, from 0 to 8 g/t inclusive, are reported:

[![](../Images/ConfTable2.png)](<javascript:void\(0\);>)

In this example:

* The complete table includes 108 records:(10 simulations + 2ETypes) * 9 cutoffs). These records show grade, volume, tonnes and contained metal (grade * tonnes) for the total model, for each simulation and cutoff.
* A **SIMNUM** of -2 gives the results for **ETYPE2**.
* A **SIMNUM** of -1 gives the results for **ETYPE1**.

The following table has been calculated using the same parameters as above. Additionally, **PCINT** =10 has been specified, resulting in percentiles being calculated from 0 to 100 in increments of 10%:

[![](../Images/ConfTable3.png)](<javascript:void\(0\);>)

Referring to row '2', column '6' in the above table: for a cutoff of 1 g/t, the 10th percentile is 2.61 g/t. This represents a 10% probability that the grade of the deposit will be below 2.61, and therefore a 90% probability that it will exceed this value.

### Input Files:

* **simmod** (Model):
  The simulated model created using the CSMODEL process.
  Required=Yes

### Output Files:

* **conf_tbl** (Table):
  Output table template for 2 model confidence tables. File names are created
  automatically from the template.
  Required=Yes

* **conf_plt** (Plot template):
  Output plot template for displaying confidence values. File names are created
  automatically from the template.
  Required=No

### Fields:

### Parameters:

* **cutint**:
  Defines the cutoff interval between successive cutoff grades.
  Range=0.00001,9999999
  Values=Undefined
  Default=1
  Required=No

* **cutmax**:
  For regular cutoff grades, this field defines the maximum cutoff grade for percentile
  tables and graphs. All simulations and the Etype estimator must have at least one value
  above the maximum cutoff value. If the selected maximum cutoff does not meet these
  criteria, then it will be automatically reduced.
  Range=0.00002,9999999
  Values=Undefined
  Default=10
  Required=No

* **pcint**:
  Defines the interval between successive percentiles in the output confidence table 2.
  Range=2.5,25
  Values='2.5', '5', '10', '20','25'
  Default=5
  Required=No

* **density**:
  The density parameter that is used for tonnage calculations.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **factor**:
  Dividing factor applied to Tonnes and Metal values before plotting - used to reduce the
  amount of annotation.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **plot_tbl**:
  Flag to specify whether a plot data table is output for every plot file created. The
  plot data table contains the data used to create the CONF_PLT plot files, and could be
  used to recreate the plot in other software such as Excel. The plot data table name is
  the same as the plot file, except that "_P" is replaced by "_T".
  Range=0,1
  Values='0' - do not output plot data tables. '1' - output all plot data tables.
  Default=0
  Required=No

* **display**:
  Flag to display whether the plot files are displayed as the process is run.
  Range=0,1
  Values='0' - do not display plot files. '1' - display plot files as the process is run.
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('modconf')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute modconf
print("Running modconf...")
dm_cmd.modconf(
    simmod_i='t_input_file',  # required
    conf_tbl_o='t_modconf_out',  # required
    # conf_plt_o='t_modconf_out',  # optional
    # cutint_p=1,  # optional
    # cutmax_p=10,  # optional
    # pcint_p=5,  # optional
    # density_p=1,  # optional
    # factor_p=1,  # optional
    # plot_tbl_p=0,  # optional
    # display_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("modconf execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_modconf_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")